# ДЗ 1. Реакция акций на анонс заседания совета директоров

**Данные:** секундные свечи и время раскрытия из файлов архива. Все часы считаются московскими; при необходимости сверить часовой пояс с поставщиком данных. Анализ оценивает связь во времени, а не доказывает причинность: на цену могли одновременно влиять рынок и другие новости.

Сначала проверяем состав и временной диапазон файлов. Затем для каждого анонса находим первую доступную свечу не раньше времени публикации. Если публикация была вне торгов, точка входа переносится на первую доступную свечу следующей сессии.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = Path('data')
TICKERS = ['GAZP', 'HNFG', 'KZOS', 'ROLO']
HORIZONS = {'1 мин': 60, '5 мин': 300, '15 мин': 900, '60 мин': 3600}

## 1. Загрузка и проверка полей

Важное ограничение: значения `volume` в выгрузке нужно трактовать согласно схеме источника данных. Код ниже считает, что это количество акций, и оценивает рублёвый оборот как `volume × цена`. Если в вашем файле объем дан в лотах или уже в рублях, поправьте ячейку нормализации перед интерпретацией ликвидности.

In [ ]:
def load_ticker(ticker):
    df = pd.read_parquet(DATA / f'{ticker}_candles.parquet')
    print(ticker, df.shape, list(df.columns))
    return df

raw = {t: load_ticker(t) for t in TICKERS}
raw['GAZP'].head()

После просмотра выведенных имён столбцов задайте их соответствия в словаре ниже. Автоопределение покрывает распространённые варианты. Если оно не сработало, укажите имя поля вручную.

In [ ]:
ALIASES = {
    'time': ['begin', 'timestamp', 'datetime', 'time', 'ts', 'date'],
    'price': ['close', 'last', 'price'],
    'volume': ['volume', 'vol', 'quantity', 'qty'],
}

def find_col(df, key):
    by_lower = {str(c).lower(): c for c in df.columns}
    for name in ALIASES[key]:
        if name in by_lower:
            return by_lower[name]
    raise KeyError(f'Не найден столбец {key}; доступны: {list(df.columns)}')

def normalize(df):
    t, p, v = (find_col(df, k) for k in ('time', 'price', 'volume'))
    out = df.copy()
    out['ts'] = pd.to_datetime(out[t], errors='coerce')
    out['price'] = pd.to_numeric(out[p], errors='coerce')
    out['volume'] = pd.to_numeric(out[v], errors='coerce')
    out = out.dropna(subset=['ts','price','volume']).sort_values('ts').drop_duplicates('ts')
    return out[['ts','price','volume']].reset_index(drop=True)

candles = {t: normalize(df) for t, df in raw.items()}
for t, d in candles.items():
    print(t, d.ts.min(), '—', d.ts.max(), 'строк:', len(d))

## 2. Событийный анализ

Доходность считаем от цены первой доступной свечи после раскрытия. Параллельно считаем изменение от последней свечи до раскрытия: оно помогает заметить, что новость могла выйти после торгов и цена на входе фактически уже относится к следующей сессии. Суммарные доходности на коротком горизонте не очищены от движения рынка.

Ликвидность вокруг точки входа оцениваем по сумме `цена × объем` за следующие 5 минут. Это наблюдавшийся оборот, а не обещание, что один участник мог купить весь объём без влияния на цену. Более консервативная оценка размера заявки — 1% и 5% от оборота окна.

In [ ]:
events = []
for ticker in TICKERS:
    disclosures = json.loads((DATA / f'{ticker}_disclosures.json').read_text(encoding='utf-8'))
    d = candles[ticker]
    for e in disclosures:
        event_ts = pd.Timestamp(f"{e['date']} {e['time']}")
        i = d['ts'].searchsorted(event_ts, side='left')
        if i >= len(d):
            events.append({'ticker':ticker, 'event_ts':event_ts, 'status':'нет последующей свечи'})
            continue
        entry = d.iloc[i]
        prev = d.iloc[i-1] if i else None
        row = {'ticker':ticker, 'event_ts':event_ts, 'entry_ts':entry.ts,
               'delay_sec':(entry.ts-event_ts).total_seconds(), 'entry_price':entry.price,
               'prev_ts':prev.ts if prev is not None else pd.NaT,
               'prev_price':prev.price if prev is not None else np.nan,
               'pre_move_pct':(entry.price/prev.price-1)*100 if prev is not None else np.nan,
               'status':'ok'}
        for label, seconds in HORIZONS.items():
            j = d['ts'].searchsorted(entry.ts + pd.Timedelta(seconds=seconds), side='left')
            if j < len(d):
                row[f'{label}_return_pct'] = (d.iloc[j].price/entry.price-1)*100
                row[f'{label}_actual_sec'] = (d.iloc[j].ts-entry.ts).total_seconds()
            else:
                row[f'{label}_return_pct'] = np.nan
                row[f'{label}_actual_sec'] = np.nan
        window = d[(d.ts >= entry.ts) & (d.ts < entry.ts + pd.Timedelta(minutes=5))]
        row['turnover_5m_rub_est'] = (window.price * window.volume).sum()
        row['volume_5m'] = window.volume.sum()
        events.append(row)

results = pd.DataFrame(events)
results

## 3. Итоги по компаниям и в среднем

Среднее показываем вместе с медианой: при 3–4 эпизодах единичное сильное движение может заметно сдвинуть среднее. Равновзвешенное среднее по событиям отвечает на вопрос о типичной реакции события; это не оценка портфельной доходности.

In [ ]:
valid = results.query("status == 'ok'").copy()
ret_cols = [f'{h}_return_pct' for h in HORIZONS]
summary = valid.groupby('ticker')[ret_cols].agg(['count','mean','median'])
summary

In [ ]:
overall = valid[ret_cols].agg(['count','mean','median']).T
overall

## 4. Оценка оборота и PnL

Потенциальный PnL показываем до комиссий, проскальзывания, налогов и ограничений на исполнение. Это сценарий «купить по цене первой свечи после раскрытия и продать через горизонт». Для практической оценки используйте более скромный доступный объём: например, 1% фактического рублёвого оборота за первые 5 минут. Без стакана невозможно оценить цену исполнения заявки и рыночное воздействие.

In [ ]:
valid['buy_1pct_turnover_rub'] = valid['turnover_5m_rub_est'] * 0.01
valid['buy_5pct_turnover_rub'] = valid['turnover_5m_rub_est'] * 0.05
for label in HORIZONS:
    r = valid[f'{label}_return_pct'] / 100
    valid[f'{label}_pnl_1pct_rub'] = valid.buy_1pct_turnover_rub * r
    valid[f'{label}_pnl_5pct_rub'] = valid.buy_5pct_turnover_rub * r
valid[['ticker','event_ts','entry_ts','turnover_5m_rub_est','buy_1pct_turnover_rub'] + [f'{h}_pnl_1pct_rub' for h in HORIZONS]]

In [ ]:
pnl_summary = valid.groupby('ticker')[[f'{h}_pnl_1pct_rub' for h in HORIZONS]].agg(['count','mean','median','sum'])
pnl_summary

## 5. График и выводы

Отрицательная/положительная доходность после анонса сама по себе не показывает, что именно анонс её вызвал. Для строгой оценки потребовались бы контрольное движение рынка, сравнимые даты без события и цены исполнения из стакана. Малое число событий также означает высокую неопределённость.

In [ ]:
for horizon, col in [('1 мин','1 мин_return_pct'), ('5 мин','5 мин_return_pct'), ('15 мин','15 мин_return_pct'), ('60 мин','60 мин_return_pct')]:
    ax = valid.plot(x='event_ts', y=col, kind='bar', figsize=(11,3), title=f'Доходность через {horizon}, %', legend=False)
    ax.axhline(0, color='black', linewidth=.8)
    plt.tight_layout(); plt.show()

# Таблица событий для включения чисел в текстовый вывод
results.to_csv('event_study_results.csv', index=False)
summary.to_csv('event_study_summary.csv')

## Шаблон итогового вывода

Заполнить после выполнения ячеек: (а) по каждому эмитенту указать медианную реакцию на горизонтах 1/5/15/60 минут и число наблюдений; (б) указать общее равновзвешенное среднее/медиану; (в) отметить случаи раскрытия вне торгов; (г) привести оценку оборота 5 минут и PnL сценария с долей участия 1%; (д) явно оговорить, что на малой выборке и без контроля рынка это описательная оценка, а не доказательство причинности.